# KoBERT + LSTM 보이스피싱 탐지 모델 (런팟 최적화)
## RTX A6000 환경 최적화된 딥러닝 학습

## 1. 환경 설정 및 GPU 최적화

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
import time
import gc
warnings.filterwarnings('ignore')

# GPU 최적화 설정
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU Name: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f}GB')
    print(f'CUDA Version: {torch.version.cuda}')
    torch.cuda.empty_cache()

## 2. 데이터 로딩 및 전처리

In [ ]:
# 훈련 데이터 로딩
print("📊 데이터 로딩 중...")
train_df = pd.read_csv("../../dataset/master_dataset_final.csv")
test_df = pd.read_csv("../../dataset/1차모델_테스트데이터셋.csv")

print(f"✅ 훈련 데이터: {len(train_df):,}개")
print(f"✅ 테스트 데이터: {len(test_df):,}개")
print(f"📈 클래스 분포 - 피싱: {train_df['is_phishing'].sum()}, 일반: {len(train_df) - train_df['is_phishing'].sum()}")

In [ ]:
import re

def clean_text(text):
    """텍스트 전처리 - 최적화된 버전"""
    text = re.sub(r'\s+', ' ', str(text))
    text = re.sub(r'[^\w\s가-힣.,!?()]', '', text)
    return text.strip()

def create_dialogue_sequences(df):
    """대화 시퀀스 생성 - 메모리 효율적"""
    dialogues = []
    
    for file_id in tqdm(df['file_id'].unique(), desc="🔄 대화 시퀀스 생성"):
        file_data = df[df['file_id'] == file_id]
        full_text = file_data['text'].iloc[0]
        
        # 효율적인 문장 분할
        sentences = [clean_text(sent) for sent in full_text.split('.') 
                    if sent.strip() and len(sent.split()) >= 2]
        
        if sentences:
            dialogues.append({
                'file_id': file_id,
                'texts': sentences[:50],  # 최대 50턴으로 제한
                'label': file_data['is_phishing'].iloc[0]
            })
    
    return dialogues

def create_test_data(df):
    """테스트 데이터 생성"""
    test_data = []
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="🔄 테스트 데이터 생성"):
        sentences = [clean_text(sent) for sent in row['text'].split('.') 
                    if sent.strip() and len(sent.split()) >= 2]
        
        if not sentences:
            sentences = [clean_text(row['text'])]
        
        test_data.append({
            'file_id': f"test_{idx}",
            'texts': sentences[:50],
            'label': row['is_phishing']
        })
    
    return test_data

# 데이터 생성
train_dialogues = create_dialogue_sequences(train_df)
test_dialogues = create_test_data(test_df)

print(f"✅ 훈련 대화: {len(train_dialogues):,}개")
print(f"✅ 테스트 대화: {len(test_dialogues):,}개")
print(f"📊 평균 문장 수: {np.mean([len(d['texts']) for d in train_dialogues]):.1f}")

# 메모리 정리
del train_df, test_df
gc.collect()

## 3. KoBERT 및 데이터셋 클래스

In [ ]:
print("🤖 KoBERT 모델 로딩...")
MODEL_NAME = "skt/kobert-base-v1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
kobert_model = AutoModel.from_pretrained(MODEL_NAME)
print("✅ KoBERT 로딩 완료")

class OptimizedDialogueDataset(Dataset):
    """GPU 메모리 효율적인 데이터셋"""
    def __init__(self, dialogues, tokenizer, max_length=128, max_turns=30):
        self.dialogues = dialogues
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.max_turns = max_turns
    
    def __len__(self):
        return len(self.dialogues)
    
    def __getitem__(self, idx):
        dialogue = self.dialogues[idx]
        texts = dialogue['texts'][:self.max_turns]
        label = dialogue['label']
        
        input_ids_list = []
        attention_mask_list = []
        
        for text in texts:
            if not text.strip():
                text = "[EMPTY]"
            
            encoded = self.tokenizer(
                f"[TURN] {text}",
                max_length=self.max_length,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )
            
            input_ids_list.append(encoded['input_ids'].squeeze(0))
            attention_mask_list.append(encoded['attention_mask'].squeeze(0))
        
        return {
            'input_ids': torch.stack(input_ids_list),
            'attention_mask': torch.stack(attention_mask_list),
            'label': torch.tensor(label, dtype=torch.long),
            'num_turns': len(texts)
        }

def optimized_collate_fn(batch):
    """메모리 효율적인 배치 처리"""
    max_turns = max(item['num_turns'] for item in batch)
    
    batch_input_ids = []
    batch_attention_mask = []
    batch_labels = []
    batch_lengths = []
    
    for item in batch:
        num_turns = item['num_turns']
        
        if num_turns < max_turns:
            pad_size = max_turns - num_turns
            pad_input_ids = torch.zeros(pad_size, item['input_ids'].size(1), dtype=torch.long)
            pad_attention_mask = torch.zeros(pad_size, item['attention_mask'].size(1), dtype=torch.long)
            
            input_ids = torch.cat([item['input_ids'], pad_input_ids], dim=0)
            attention_mask = torch.cat([item['attention_mask'], pad_attention_mask], dim=0)
        else:
            input_ids = item['input_ids']
            attention_mask = item['attention_mask']
        
        batch_input_ids.append(input_ids)
        batch_attention_mask.append(attention_mask)
        batch_labels.append(item['label'])
        batch_lengths.append(num_turns)
    
    return {
        'input_ids': torch.stack(batch_input_ids),
        'attention_mask': torch.stack(batch_attention_mask),
        'labels': torch.stack(batch_labels),
        'lengths': torch.tensor(batch_lengths, dtype=torch.long)
    }

print("✅ 데이터셋 클래스 정의 완료")

## 4. 최적화된 모델 정의

In [ ]:
class OptimizedPhishingDetector(nn.Module):
    """RTX A6000 최적화된 모델"""
    def __init__(self, kobert_model, hidden_size=512, num_classes=2, dropout=0.2):
        super(OptimizedPhishingDetector, self).__init__()
        
        self.kobert = kobert_model
        self.kobert_hidden_size = kobert_model.config.hidden_size
        
        # 효율적인 문장 임베딩
        self.sentence_projection = nn.Sequential(
            nn.Linear(self.kobert_hidden_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout)
        )
        
        # 양방향 LSTM
        self.dialogue_lstm = nn.LSTM(
            hidden_size, 
            hidden_size // 2, 
            num_layers=2,
            batch_first=True, 
            bidirectional=True,
            dropout=dropout
        )
        
        # Attention 메커니즘
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_size,
            num_heads=8,
            dropout=dropout,
            batch_first=True
        )
        
        self.attention_norm = nn.LayerNorm(hidden_size)
        
        # 분류기
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.LayerNorm(hidden_size // 2),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout // 2),
            nn.Linear(hidden_size // 2, num_classes)
        )
        
        self._init_weights()
    
    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
    
    def forward(self, input_ids, attention_mask, lengths):
        batch_size, max_turns, seq_len = input_ids.size()
        
        # BERT 임베딩
        input_ids_flat = input_ids.view(-1, seq_len)
        attention_mask_flat = attention_mask.view(-1, seq_len)
        
        with torch.cuda.amp.autocast():  # Mixed precision
            kobert_outputs = self.kobert(
                input_ids=input_ids_flat,
                attention_mask=attention_mask_flat
            )
        
        sentence_embeddings = kobert_outputs.last_hidden_state[:, 0, :]
        sentence_embeddings = sentence_embeddings.view(batch_size, max_turns, -1)
        
        # 문장 특성 추출
        sentence_features = self.sentence_projection(sentence_embeddings)
        
        # LSTM 처리
        lstm_out, _ = self.dialogue_lstm(sentence_features)
        
        # Attention
        padding_mask = torch.arange(max_turns, device=lengths.device).expand(
            batch_size, max_turns
        ) >= lengths.unsqueeze(1)
        
        attended_out, _ = self.attention(
            lstm_out, lstm_out, lstm_out,
            key_padding_mask=padding_mask
        )
        
        # Residual connection
        attended_out = self.attention_norm(lstm_out + attended_out)
        
        # 마스킹 및 풀링
        mask = ~padding_mask.unsqueeze(-1)
        masked_attended = attended_out * mask
        dialogue_repr = masked_attended.sum(dim=1) / lengths.unsqueeze(-1).float()
        
        # 분류
        logits = self.classifier(dialogue_repr)
        
        return logits

# 모델 초기화
print("🏗️ 모델 생성 중...")
model = OptimizedPhishingDetector(kobert_model, hidden_size=512, dropout=0.2)
model.to(device)
print(f"✅ 모델 파라미터 수: {sum(p.numel() for p in model.parameters()):,}")

# 메모리 정리
torch.cuda.empty_cache()

## 5. 훈련 설정 및 함수

In [ ]:
# 하이퍼파라미터 (RTX A6000 최적화)
BATCH_SIZE = 24  # A6000의 48GB VRAM 활용
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 8
WARMUP_RATIO = 0.1

print(f"⚙️ 하이퍼파라미터:")
print(f"   배치 크기: {BATCH_SIZE}")
print(f"   학습률: {LEARNING_RATE}")
print(f"   에포크: {NUM_EPOCHS}")

# 데이터셋 및 로더 생성
train_dataset = OptimizedDialogueDataset(train_dialogues, tokenizer)
test_dataset = OptimizedDialogueDataset(test_dialogues, tokenizer)

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    collate_fn=optimized_collate_fn,
    num_workers=4,  # 멀티프로세싱
    pin_memory=True  # GPU 전송 최적화
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    collate_fn=optimized_collate_fn,
    num_workers=4,
    pin_memory=True
)

print(f"✅ 훈련 배치 수: {len(train_loader)}")
print(f"✅ 테스트 배치 수: {len(test_loader)}")

In [ ]:
# 손실 함수 및 옵티마이저
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(
    model.parameters(), 
    lr=LEARNING_RATE, 
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.999)
)

# 학습률 스케줄러
total_steps = len(train_loader) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LEARNING_RATE,
    total_steps=total_steps,
    pct_start=WARMUP_RATIO,
    anneal_strategy='cos'
)

# Mixed Precision Scaler
scaler = torch.cuda.amp.GradScaler()

print("✅ 옵티마이저 및 스케줄러 설정 완료")

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, scheduler, scaler, epoch, device):
    """최적화된 훈련 함수"""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    # 진행도 표시
    pbar = tqdm(train_loader, desc=f"🚀 Epoch {epoch+1}/{NUM_EPOCHS} 훈련")
    
    for batch_idx, batch in enumerate(pbar):
        input_ids = batch['input_ids'].to(device, non_blocking=True)
        attention_mask = batch['attention_mask'].to(device, non_blocking=True)
        labels = batch['labels'].to(device, non_blocking=True)
        lengths = batch['lengths'].to(device, non_blocking=True)
        
        optimizer.zero_grad()
        
        # Mixed Precision Training
        with torch.cuda.amp.autocast():
            logits = model(input_ids, attention_mask, lengths)
            loss = criterion(logits, labels)
        
        # Backward pass
        scaler.scale(loss).backward()
        
        # Gradient clipping
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        # 통계 업데이트
        total_loss += loss.item()
        _, predicted = torch.max(logits.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        # 진행도 업데이트
        if batch_idx % 10 == 0:
            current_acc = 100. * correct / total
            current_loss = total_loss / (batch_idx + 1)
            lr = scheduler.get_last_lr()[0]
            pbar.set_postfix({
                'Loss': f'{current_loss:.4f}',
                'Acc': f'{current_acc:.2f}%',
                'LR': f'{lr:.2e}'
            })
        
        # 메모리 정리
        if batch_idx % 50 == 0:
            torch.cuda.empty_cache()
    
    avg_loss = total_loss / len(train_loader)
    accuracy = 100. * correct / total
    
    return avg_loss, accuracy

def validate_epoch(model, val_loader, criterion, device):
    """검증 함수"""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_predictions = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        pbar = tqdm(val_loader, desc="📊 검증 중")
        for batch in pbar:
            input_ids = batch['input_ids'].to(device, non_blocking=True)
            attention_mask = batch['attention_mask'].to(device, non_blocking=True)
            labels = batch['labels'].to(device, non_blocking=True)
            lengths = batch['lengths'].to(device, non_blocking=True)
            
            with torch.cuda.amp.autocast():
                logits = model(input_ids, attention_mask, lengths)
                loss = criterion(logits, labels)
            
            total_loss += loss.item()
            probs = torch.softmax(logits, dim=1)
            _, predicted = torch.max(logits.data, 1)
            
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    avg_loss = total_loss / len(val_loader)
    accuracy = 100. * correct / total
    
    return avg_loss, accuracy, all_predictions, all_labels, all_probs

print("✅ 훈련 함수 정의 완료")

## 6. 모델 훈련

In [ ]:
print("🎯 모델 훈련 시작!")
print(f"⏰ 예상 훈련 시간: {NUM_EPOCHS * len(train_loader) * BATCH_SIZE / 1000:.0f}분")
print("="*60)

# 훈련 기록
train_losses = []
train_accs = []
best_acc = 0
start_time = time.time()

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()
    
    # 훈련
    train_loss, train_acc = train_epoch(
        model, train_loader, criterion, optimizer, scheduler, scaler, epoch, device
    )
    
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    
    epoch_time = time.time() - epoch_start
    total_time = time.time() - start_time
    
    # 결과 출력
    print(f"\n📈 Epoch {epoch+1}/{NUM_EPOCHS} 완료:")
    print(f"   훈련 손실: {train_loss:.4f}")
    print(f"   훈련 정확도: {train_acc:.2f}%")
    print(f"   소요 시간: {epoch_time:.1f}초")
    print(f"   총 시간: {total_time/60:.1f}분")
    print(f"   현재 학습률: {scheduler.get_last_lr()[0]:.2e}")
    
    # 베스트 모델 저장
    if train_acc > best_acc:
        best_acc = train_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_acc': train_acc,
            'train_loss': train_loss
        }, '../../models/best_kobert_lstm_model.pth')
        print(f"💾 베스트 모델 저장 (정확도: {best_acc:.2f}%)")
    
    print("-" * 60)
    
    # GPU 메모리 정리
    torch.cuda.empty_cache()

total_training_time = time.time() - start_time
print(f"\n🎉 훈련 완료!")
print(f"⏱️ 총 훈련 시간: {total_training_time/60:.1f}분")
print(f"🏆 최고 훈련 정확도: {best_acc:.2f}%")

## 7. 모델 평가

In [ ]:
print("📊 모델 평가 시작...")

# 베스트 모델 로드
checkpoint = torch.load('../../models/best_kobert_lstm_model.pth')
model.load_state_dict(checkpoint['model_state_dict'])
print(f"✅ 베스트 모델 로드 완료 (Epoch {checkpoint['epoch']+1})")

# 테스트 평가
test_loss, test_acc, test_predictions, test_labels, test_probs = validate_epoch(
    model, test_loader, criterion, device
)

print(f"\n🎯 최종 테스트 결과:")
print(f"   테스트 손실: {test_loss:.4f}")
print(f"   테스트 정확도: {test_acc:.2f}%")

# ROC AUC 계산
test_probs_array = np.array(test_probs)[:, 1]
fpr, tpr, _ = roc_curve(test_labels, test_probs_array)
roc_auc = auc(fpr, tpr)
print(f"   AUC 점수: {roc_auc:.4f}")

# 상세 분류 리포트
print(f"\n📋 상세 성능 분석:")
print(classification_report(test_labels, test_predictions, target_names=['Normal', 'Phishing']))

## 8. 결과 시각화

In [ ]:
# 한글 폰트 설정
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

# 결과 시각화
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))

# 1. 훈련 손실 및 정확도
epochs = range(1, len(train_losses) + 1)
ax1.plot(epochs, train_losses, 'b-', label='Train Loss', linewidth=2)
ax1_twin = ax1.twinx()
ax1_twin.plot(epochs, train_accs, 'r-', label='Train Accuracy', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss', color='b')
ax1_twin.set_ylabel('Accuracy (%)', color='r')
ax1.set_title('Training Progress')
ax1.grid(True, alpha=0.3)

# 2. Confusion Matrix
cm = confusion_matrix(test_labels, test_predictions)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax2,
            xticklabels=['Normal', 'Phishing'],
            yticklabels=['Normal', 'Phishing'])
ax2.set_title(f'Confusion Matrix (Acc: {test_acc:.2f}%)')
ax2.set_xlabel('Predicted')
ax2.set_ylabel('Actual')

# 3. ROC Curve
ax3.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
ax3.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
ax3.set_xlim([0.0, 1.0])
ax3.set_ylim([0.0, 1.05])
ax3.set_xlabel('False Positive Rate')
ax3.set_ylabel('True Positive Rate')
ax3.set_title('ROC Curve')
ax3.legend(loc="lower right")
ax3.grid(True, alpha=0.3)

# 4. 클래스별 성능
from sklearn.metrics import precision_recall_fscore_support
precision, recall, f1, support = precision_recall_fscore_support(test_labels, test_predictions)

metrics = ['Precision', 'Recall', 'F1-Score']
normal_scores = [precision[0], recall[0], f1[0]]
phishing_scores = [precision[1], recall[1], f1[1]]

x = np.arange(len(metrics))
width = 0.35

ax4.bar(x - width/2, normal_scores, width, label='Normal', alpha=0.7, color='blue')
ax4.bar(x + width/2, phishing_scores, width, label='Phishing', alpha=0.7, color='red')
ax4.set_xlabel('Metrics')
ax4.set_ylabel('Score')
ax4.set_title('Class-wise Performance')
ax4.set_xticks(x)
ax4.set_xticklabels(metrics)
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../../models/kobert_lstm_results.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ 결과 시각화 완료")

## 9. 최종 모델 저장

In [ ]:
# 최종 모델 저장
final_model_path = '../../models/kobert_lstm_final.pth'
torch.save({
    'model_state_dict': model.state_dict(),
    'model_config': {
        'hidden_size': 512,
        'num_classes': 2,
        'dropout': 0.2
    },
    'tokenizer_name': MODEL_NAME,
    'training_config': {
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'num_epochs': NUM_EPOCHS,
        'weight_decay': WEIGHT_DECAY
    },
    'results': {
        'best_train_acc': best_acc,
        'test_acc': test_acc,
        'test_loss': test_loss,
        'auc': roc_auc,
        'precision': precision.tolist(),
        'recall': recall.tolist(),
        'f1': f1.tolist()
    },
    'training_history': {
        'train_losses': train_losses,
        'train_accs': train_accs
    }
}, final_model_path)

print(f"💾 최종 모델 저장 완료: {final_model_path}")

# 최종 결과 요약
print("\n" + "="*60)
print("🎉 KoBERT + LSTM 보이스피싱 탐지 모델 훈련 완료")
print("="*60)
print(f"📊 모델 아키텍처: KoBERT + Bidirectional LSTM + Attention")
print(f"🔧 파라미터 수: {sum(p.numel() for p in model.parameters()):,}")
print(f"📈 훈련 데이터: {len(train_dialogues):,} 대화")
print(f"📊 테스트 데이터: {len(test_dialogues):,} 대화")
print(f"")
print(f"🏆 최종 성능:")
print(f"   테스트 정확도: {test_acc:.2f}%")
print(f"   AUC 점수: {roc_auc:.4f}")
print(f"   Normal - Precision: {precision[0]:.3f}, Recall: {recall[0]:.3f}, F1: {f1[0]:.3f}")
print(f"   Phishing - Precision: {precision[1]:.3f}, Recall: {recall[1]:.3f}, F1: {f1[1]:.3f}")
print(f"")
print(f"⏱️ 총 훈련 시간: {total_training_time/60:.1f}분")
print(f"🎯 GPU 활용: RTX A6000 최적화 완료")
print("="*60)

# GPU 메모리 정리
torch.cuda.empty_cache()
print("🧹 GPU 메모리 정리 완료")